# NB02 – Data Transformation

## Purpose

In NB01 I collected the raw product data and saved one JSON file per country in
the `data/raw/` folder. Those files are the raw API responses, which are nested
and not easy to analyse directly. Each file holds a list of products under a key
called `hits`, and inside every product the nutritional values sit in their own
dictionary called `nutriments`.

The goal of this notebook is to turn that raw data into a clean, flat table that
I can actually work with. For each country I go through every product in the file
and pull out the fields I need:

- the **barcode** (`code`)
- the **product name** (`product_name`)
- the **brand** (`brands`)
- the **sugar content** in grams per 100g (`sugars_100g`, which sits inside `nutriments`)
- the **country** the product was collected for

The country is the one column that does not come from the product record itself.
Each API request already filtered to a single country, so the response never
repeats that information product by product. I take it from the name of the file
each set of products was saved in, and stamp it onto every row.

The other thing this notebook has to handle is missing data. Many products in
Open Food Facts have been scanned but never filled in, so some have no name, no
brand, or no sugar value at all. I do not remove these rows here. They are kept
so that the amount of missing data can be counted per country in NB03, since how
much is missing is itself part of the answer.

Once everything is pulled out, I save the result as a **CSV file** in the
`data/processed/` folder. That CSV becomes the input for NB03, where I do the
actual analysis (comparing sugar content between countries with and without a
sugar tax).

I do not do any analysis here. This notebook only reshapes the data from nested
JSON into a clean table.

In [ ]:
import json 
import pandas as pd 
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

rows = []
for path in RAW_DIR.glob("*.json"): 
    country = path.stem # .stem basically just takes the name of the file without the type of file  so it stores "italy"
    with open(RAW_DIR / f"{country}.json",  encoding='utf-8') as f: 
        sample = json.load(f)

        for reading in sample["hits"]: 
            # Some products were scanned but never filled in, so fields can be missing.
            # .get() returns None instead of crashing, and the {} and [] are fallbacks
            # so the next lines still have something to work with.
            nutriments = reading.get("nutriments", {})
            brands = reading.get("brands", [])

            rows.append({
                "country": country,
                "barcode": reading.get("code"),
                "product_name": reading.get("product_name"),
                "brand": brands[0] if brands else None, # Will pass as None if there is no value for sugar
                "sugar_content_per_100g": nutriments.get("sugars_100g")
            })

df = pd.DataFrame(rows)

In [ ]:
# After generating the Data Frame, I want to have a look at the structure 
df.head()

,country,barcode,product_name,brand,sugar_content_per_100g
0,italy,0000503210227,NaN,NaN,NaN
1,italy,8002516010223,La classica,Tomarchio,11.0
2,italy,3270190005261,PULP' Saveur Orange,Carrefour,8.4
3,italy,4060800129680,Pepsi lemon,Pepsi,10.7
4,italy,4060800001771,Pepsi-cola,pepsi,10.9


In [12]:
# I am just realising I have some entried with NaN values for sugar content. 
# As I have 100 entries per country, I want to know how many have values for each country
# I am going to use a .groupby to figure it out 
print(df.groupby("country")["sugar_content_per_100g"].agg(["count", "size"])) 

                count  size
country                    
france             94   100
germany            90   100
italy              97   100
united-kingdom     80   100


In [ ]:
# I am going to create a new DataFrame that filters out all the values that are NaN for sugar contents. 
# For this, I am going to use .notna() 
df_clean = df[df["sugar_content_per_100g"].notna()]
df_clean.head()

,country,barcode,product_name,brand,sugar_content_per_100g
1,italy,8002516010223,La classica,Tomarchio,11.0
2,italy,3270190005261,PULP' Saveur Orange,Carrefour,8.4
3,italy,4060800129680,Pepsi lemon,Pepsi,10.7
4,italy,4060800001771,Pepsi-cola,pepsi,10.9
5,italy,5449000005090,Fanta,fanta,11.8


In [11]:
# Now I save it into a clean data CSV 
df_clean.to_csv(PROCESSED_DIR / "sodas_clean.csv", index=False, encoding='utf-8')

### Filtering out the missing sugar values

Looking at the first rows of the table, the very first product has no name, no
brand and no sugar value. It is a real record in Open Food Facts: someone scanned
the barcode and never filled anything in. Records like this are common in a
database built by volunteers, so the table needs to deal with them rather than
assume every product is complete.

Before removing anything, I counted how many products each country has and how
many of those have a sugar value, using `.groupby()` with `count` and `size`.
`size` counts every row, while `count` ignores missing values, so the difference
between them is the number of products with no sugar recorded:

| Country        | Products collected | With a sugar value | Missing |
|----------------|--------------------|--------------------|---------|
| France         | 100                | 94                 | 6       |
| Germany        | 100                | 90                 | 10      |
| Italy          | 100                | 97                 | 3       |
| United Kingdom | 100                | 80                 | 20      |

Every country has 100 products because that is the number I requested per
request, so the groups start out the same size. What differs is how complete the
records are. The United Kingdom is missing a fifth of its sugar values, which is
noticeably worse than the other three, and the United Kingdom is one of the two
countries with a sugar tax. This matters, because if the products missing their
sugar value are not a random selection of British soft drinks, the average I
calculate for the UK is based on a slightly different set of drinks than the
average for Italy.

I then filtered the table with `.notna()` to keep only the rows where
`sugar_content_per_100g` is present. I deliberately did not drop every row with
any missing value, because a product with no name or no brand is still usable as
long as it has a sugar value, and sugar is the only field the analysis needs.

This leaves group sizes of 94, 90, 97 and 80. These are close enough that they do
not cause a problem for comparing averages. An average does not require equal
group sizes to be valid: the mean of 80 values is a perfectly good estimate, it is
just based on slightly less information than the mean of 97, so it is a little
less precise. Group size only becomes misleading if I plot raw counts rather than
averages or proportions, since a country with more products would appear larger
simply for having more rows. I therefore report the number of products alongside
every average, and use proportions rather than counts in any chart.

The filtered table is saved as a CSV in `data/processed/`, which becomes the input
for NB03.